In [ ]:
# Fix Cell 2 — Stable Colab Python 3.12 package reset
# Run this once, then runtime will restart.

!pip uninstall -y numpy scipy scikit-learn pandas pyarrow sentence-transformers faiss-cpu bm25s rank-bm25

!pip install -q --no-cache-dir \
    numpy==1.26.4 \
    scipy==1.12.0 \
    pandas==2.2.2 \
    scikit-learn==1.4.2 \
    pyarrow==16.1.0

!pip install -q --no-cache-dir \
    torch \
    transformers==4.44.2 \
    sentence-transformers==3.0.1 \
    faiss-cpu==1.8.0 \
    rank-bm25==0.2.2 \
    bm25s==0.2.5 \
    tqdm regex

import os
os.kill(os.getpid(), 9)

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
Found existing installation: pandas 3.0.3
Uninstalling pandas-3.0.3:
  Successfully uninstalled pandas-3.0.3
Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: sentence-transformers 5.5.0
Uninstalling sentence-transformers-5.5.0:
  Successfully uninstalled sentence-transformers-5.5.0
Found existing installation: faiss-cpu 1.13.2
Uninstalling faiss-cpu-1.13.2:
  Successfully uninstalled faiss-cpu-1.13.2
Found existing installation: bm25s 0.3.9
Uninstalling bm25s-0.3.9:
  Successfully uninstalled bm25s-0.3.9
Found existing installation: rank-bm25 0.2.2
Uninstal

In [ ]:
# Sanity check after restart

import numpy as np
import pandas as pd
import scipy
import sklearn
import pyarrow
import faiss
from sentence_transformers import SentenceTransformer

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("pyarrow:", pyarrow.__version__)
print("faiss:", faiss.__version__)

# Quick pandas test
tmp = pd.DataFrame({"a": [1, 2], "b": ["x", "y"]})
display(tmp)

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


numpy: 1.26.4
pandas: 2.2.2
scipy: 1.12.0
sklearn: 1.4.2
pyarrow: 16.1.0
faiss: 1.8.0


,a,b
0,1,x
1,2,y


In [ ]:
# Cell 1 — Environment setup + persistent Google Drive workspace
# Purpose:
# - Install only the libraries needed for Sprint-1.
# - Mount Google Drive so expensive artifacts survive Colab restarts.
# - Create a fixed project directory for all saved outputs.

!pip -q install -U \
    pandas numpy tqdm regex pyarrow \
    sentence-transformers faiss-cpu \
    rank-bm25 bm25s

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, time, gc, re, math, pickle, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Change this folder name if you want.
PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")

DATA_DIR = Path("/content")
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
INDEX_DIR = PROJECT_DIR / "indexes"
LOG_DIR = PROJECT_DIR / "logs"

for d in [PROJECT_DIR, ARTIFACT_DIR, INDEX_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Input files expected in Colab /content.
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "sample submission.csv"
KB_PATH = DATA_DIR / "Knowledge_Base.txt"

# Saved artifacts for Sprint-1.
CLEAN_KB_PATH = ARTIFACT_DIR / "clean_kb.txt"
CHUNK_PATH = ARTIFACT_DIR / "chunks.jsonl"
CHUNK_META_PATH = ARTIFACT_DIR / "chunk_metadata.parquet"
CONFIG_PATH = ARTIFACT_DIR / "sprint1_config.json"

FAISS_PATH = INDEX_DIR / "bge_m3_faiss.index"
EMBED_META_PATH = INDEX_DIR / "dense_index_meta.pkl"

BM25_DIR = INDEX_DIR / "bm25s_index"
BM25_CORPUS_PATH = INDEX_DIR / "bm25_corpus.jsonl"

# Sprint-1 config
CONFIG = {
    "project_dir": str(PROJECT_DIR),
    "chunk_tokens": 420,
    "chunk_overlap_tokens": 70,
    "min_chunk_tokens": 60,
    "dense_model": "BAAI/bge-m3",
    "random_seed": 42,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

print("Setup complete.")
print("Project directory:", PROJECT_DIR)
print("Artifacts directory:", ARTIFACT_DIR)
print("Indexes directory:", INDEX_DIR)

print("\nExpected input files:")
for p in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH, KB_PATH]:
    print(p.name, "FOUND" if p.exists() else "MISSING")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.12.0 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.12.0 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is i

In [ ]:
# Cell 2 — Load train/test/sample CSV files and detect important columns
# Purpose:
# - Load train.csv, test.csv, and sample submission.csv.
# - Automatically detect index/question/answer/reasoning columns.
# - Save a small dataset summary to Drive for Sprint-2 reference.

import pandas as pd
import json
from pathlib import Path

# Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

print("train:", train_df.shape, train_df.columns.tolist())
print("test:", test_df.shape, test_df.columns.tolist())
print("sample:", sample_df.shape, sample_df.columns.tolist())

def find_col(df, candidates, required=True):
    """
    Finds a column from candidate names.
    Works with exact lowercase match first, then substring match.
    """
    lower_map = {str(c).lower().strip(): c for c in df.columns}

    for cand in candidates:
        key = cand.lower().strip()
        if key in lower_map:
            return lower_map[key]

    for c in df.columns:
        lc = str(c).lower().strip()
        for cand in candidates:
            if cand.lower().strip() in lc:
                return c

    if required:
        raise ValueError(f"Could not find any of {candidates} in columns {df.columns.tolist()}")
    return None

# Detect columns
ID_COL_TRAIN = find_col(train_df, ["index", "id"])
Q_COL_TRAIN = find_col(train_df, ["question", "প্রশ্ন"])
A_COL_TRAIN = find_col(train_df, ["answer", "উত্তর"])
R_COL_TRAIN = find_col(train_df, ["reasoning", "rationale", "explanation", "ব্যাখ্যা"], required=False)

ID_COL_TEST = find_col(test_df, ["index", "id"])
Q_COL_TEST = find_col(test_df, ["question", "প্রশ্ন"])

ID_COL_SUB = find_col(sample_df, ["index", "id"])
A_COL_SUB = find_col(sample_df, ["answer", "উত্তর"])

print("\nDetected columns:")
print("TRAIN:", ID_COL_TRAIN, Q_COL_TRAIN, A_COL_TRAIN, R_COL_TRAIN)
print("TEST:", ID_COL_TEST, Q_COL_TEST)
print("SUB:", ID_COL_SUB, A_COL_SUB)

# Save dataset metadata for later sessions
DATASET_INFO = {
    "train_shape": train_df.shape,
    "test_shape": test_df.shape,
    "sample_shape": sample_df.shape,
    "train_columns": train_df.columns.tolist(),
    "test_columns": test_df.columns.tolist(),
    "sample_columns": sample_df.columns.tolist(),
    "ID_COL_TRAIN": ID_COL_TRAIN,
    "Q_COL_TRAIN": Q_COL_TRAIN,
    "A_COL_TRAIN": A_COL_TRAIN,
    "R_COL_TRAIN": R_COL_TRAIN,
    "ID_COL_TEST": ID_COL_TEST,
    "Q_COL_TEST": Q_COL_TEST,
    "ID_COL_SUB": ID_COL_SUB,
    "A_COL_SUB": A_COL_SUB,
}

DATASET_INFO_PATH = ARTIFACT_DIR / "dataset_info.json"
with open(DATASET_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(DATASET_INFO, f, ensure_ascii=False, indent=2, default=str)

print("\nSaved dataset info:", DATASET_INFO_PATH)

display(train_df.head())
display(test_df.head())
display(sample_df.head())

train: (3000, 4) ['index', 'question', 'answer', 'reasoning']
test: (1500, 2) ['index', 'question']
sample: (9, 2) ['index', 'answer']

Detected columns:
TRAIN: index question answer reasoning
TEST: index question
SUB: index answer

Saved dataset info: /content/drive/MyDrive/bangla_qa_070_sprint/artifacts/dataset_info.json


,index,question,answer,reasoning
0,train_0001,মুক্তিযুদ্ধে অবদানের জন্য আবু সালেককে কোন সম্ম...,বীর প্রতীক।,আবু সালেক মুক্তিযুদ্ধে তার অসাধারণ সাহস ও বীরত...
1,train_0002,ফিলিস্তিনি ইসলামি জিহাদের উদ্দেশ্য কী?,একটি সার্বভৌম ইসলামী ফিলিস্তিনি রাষ্ট্র প্রতিষ...,ফিলিস্তিনি ইসলামি জিহাদের উদ্দেশ্য হলো একটি সা...
2,train_0003,ত্রিপুরার কংগ্রেস-টিইউজেএস জোট সরকার কবে গঠিত ...,"৫ ফেব্রুয়ারি, ১৯৮৮।",সুধীর রঞ্জন মজুমদার ১৯৮৮ সালের ৫ ফেব্রুয়ারিতে...
3,train_0004,লীলা সুমন্ত মূলগাওকারকে ভারত সরকার কোন সম্মানে...,পদ্মশ্রী।,"প্রদত্ত তথ্য অনুসারে, লীলা সুমন্ত মূলগাওকারকে ..."
4,train_0005,কালা পানি চলচ্চিত্রের সঙ্গীতায়োজন করেছেন কে?,শচীন দেববর্মণ।,"প্রদত্ত তথ্য অনুসারে, ""কালা পানি"" চলচ্চিত্রের ..."


,index,question
0,test_0001,অমলক রতন কোহলি কোন ক্ষেত্রে বিশিষ্ট হিসেবে সম্...
1,test_0002,ম্যাক্স ব্যারনের বন্ধুরা তাকে কেন প্রায়ই ঠাট্...
2,test_0003,ব্রিটিশ আমলে ব্যাঙ্কশাল কোর্ট কী নামে পরিচিত ছিল?
3,test_0004,কে জুতা বানানোর আধুনিক যন্ত্র তৈরি করেন?
4,test_0005,সার্ভাইভার সিরিজের ম্যাচে কখনও কখনও কী ধরনের অ...


,index,answer
0,test_0001,সঠিক উত্তর
1,test_0002,সঠিক উত্তর
2,test_0003,সঠিক উত্তর
3,test_0004,সঠিক উত্তর
4,test_0005,সঠিক উত্তর


In [ ]:
# Cell 3 — Bangla normalization + scoring + cleaning helpers
# Purpose:
# - Define reusable functions for Bangla text normalization.
# - Define token-level F1 for validation.
# - Define basic Wikipedia boilerplate/noise removal.
# - These functions will be used by chunking, retrieval validation, and Sprint-2 QA.

import re
from collections import Counter

# Bangla + English digit maps
BN_TO_EN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
EN_TO_BN_DIGITS = str.maketrans("0123456789", "০১২৩৪৫৬৭۸৯")

BANGLA_QWORDS = {
    "কি", "কী", "কে", "কার", "কাকে", "কাদের", "কোথায়", "কোথায়",
    "কখন", "কবে", "কেন", "কিভাবে", "কীভাবে", "কোন", "কোনটি",
    "কত", "কতজন", "কিসের", "কিসে"
}

BANGLA_STOPWORDS_LIGHT = {
    "কি", "কী", "কে", "কার", "কাকে", "কাদের", "কোথায়", "কোথায়",
    "কখন", "কবে", "কেন", "কিভাবে", "কীভাবে", "কোন", "কোনটি",
    "কত", "কতজন", "কিসের", "কিসে", "হয়", "হয়", "হলো", "ছিল",
    "করেন", "করে", "করেছিল", "প্রদান", "করা", "হয়েছিল", "হয়েছিল",
    "নাম", "বলা", "হিসেবে", "জন্য", "এর", "এবং", "ও", "তে", "থেকে",
    "ছিলেন", "হলেন", "হয়েছে", "হয়েছে", "করা হয়", "করা হয়"
}

def normalize_bn_text(text):
    """
    Light Bangla-safe text normalization.
    Does not aggressively stem or alter words.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove invisible Unicode characters often found in Bangla scraped text
    text = text.replace("\ufeff", " ")
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    text = text.replace("\xa0", " ")

    # Normalize punctuation variants
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("“", "\"").replace("”", "\"")
    text = text.replace("‘", "'").replace("’", "'")

    # Normalize spaces/newlines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def normalize_for_match(text):
    """
    Normalization for token overlap/F1/retrieval matching.
    Converts Bangla digits to English digits and removes most punctuation.
    """
    text = normalize_bn_text(text).lower()
    text = text.translate(BN_TO_EN_DIGITS)

    # Keep Bangla/English alnum tokens, replace punctuation with spaces
    text = re.sub(r"[।,;:!?\"'()\[\]{}<>/\\|+=*_~`।\-]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def bn_word_tokenize(text):
    """
    Tokenizer for scoring and BM25.
    Keeps Bangla words, English words, and numbers.
    """
    text = normalize_for_match(text)
    return text.split()

def token_f1(pred, gold):
    """
    Token-level F1 used for local validation.
    Macro average over examples will be computed later.
    """
    pred_tokens = bn_word_tokenize(pred)
    gold_tokens = bn_word_tokenize(gold)

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    pc = Counter(pred_tokens)
    gc = Counter(gold_tokens)
    common = sum((pc & gc).values())

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

def answer_postprocess(ans):
    """
    Basic short-answer cleanup.
    Mainly used for train answers and later predicted answers.
    """
    ans = normalize_bn_text(ans)

    ans = re.sub(
        r"^(উত্তর|চূড়ান্ত উত্তর|চূড়ান্ত উত্তর|সংক্ষিপ্ত উত্তর)\s*[:：\-]\s*",
        "",
        ans
    )

    ans = re.sub(r"\[\s*[০-৯0-9]+\s*\]", "", ans)
    ans = ans.strip(" \"'।.,;:-")

    if not ans:
        return ""

    # Keep final Bangla punctuation
    return ans + "।"

def clean_wiki_noise(text):
    """
    Removes common Wikipedia UI/navigation boilerplate.
    This is intentionally conservative to avoid deleting real content.
    """
    text = normalize_bn_text(text)

    noise_phrases = [
        "বিষয়বস্তুতে চলুন",
        "প্রধান মেনু",
        "পার্শ্বদণ্ডে নিন",
        "লুকান",
        "অনুসন্ধান",
        "অ্যাকাউন্ট তৈরি করুন",
        "প্রবেশ করুন",
        "সম্পাদনা",
        "ইতিহাস দেখুন",
        "সরঞ্জাম",
        "মুদ্রণযোগ্য সংস্করণ",
        "গোপনীয়তার নীতি",
        "উইকিপিডিয়া বৃত্তান্ত",
        "দাবিত্যাগ",
        "মোবাইল সংস্করণ",
        "আলোচনা যোগ করুন",
        "ক্রিয়েটিভ কমন্স অ্যাট্রিবিউশন",
        "ব্যবহারের শর্তাবলী",
        "গোপনীয়তা নীতির",
        "উইকিমিডিয়া ফাউন্ডেশন",
    ]

    for phrase in noise_phrases:
        text = text.replace(phrase, " ")

    # Remove citation markers like [১], [ ১২ ], etc.
    text = re.sub(r"\[\s*[০-৯0-9]+\s*\]", " ", text)

    # Remove repeated menu-ish short lines
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            lines.append("")
            continue

        # Drop extremely common UI fragments or very short menu fragments
        if line in {"বাংলা", "পড়ুন", "সম্পাদনা", "ইতিহাস দেখুন", "তথ্যসূত্র", "বহিঃসংযোগ"}:
            continue

        lines.append(line)

    text = "\n".join(lines)

    # Collapse whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()

# Smoke test
print("Normalization test:")
print(normalize_for_match("বীর প্রতীক। [১]"))
print("F1 test:", token_f1("বীর প্রতীক।", "বীর প্রতীক।"))
print("Answer postprocess:", answer_postprocess(" বীর প্রতীক "))

Normalization test:
বীর প্রতীক 1
F1 test: 1.0
Answer postprocess: বীর প্রতীক।


In [ ]:
# Cell 4 — Clean raw Knowledge_Base.txt and save cleaned KB
# Purpose:
# - Read the large Bangla KB safely.
# - Apply Wikipedia boilerplate removal and Bangla normalization.
# - Save cleaned_kb.txt to Google Drive so Sprint-2/next runs never need to redo it.
# RAM-safe:
# - Reads once, cleans once, writes to Drive.
# - If cleaned_kb.txt already exists, loads only basic stats and skips recomputation.

import os, gc, time
from pathlib import Path

start_time = time.time()

if CLEAN_KB_PATH.exists():
    print("Cleaned KB already exists. Skipping cleaning.")
    print("Path:", CLEAN_KB_PATH)
    print("Size MB:", round(CLEAN_KB_PATH.stat().st_size / (1024 * 1024), 2))
else:
    print("Reading raw KB:", KB_PATH)

    with open(KB_PATH, "r", encoding="utf-8", errors="ignore") as f:
        raw_kb = f.read()

    print("Raw KB chars:", len(raw_kb))
    print("Raw KB size MB:", round(len(raw_kb.encode("utf-8")) / (1024 * 1024), 2))

    print("Cleaning KB...")
    clean_kb = clean_wiki_noise(raw_kb)

    print("Cleaned KB chars:", len(clean_kb))
    print("Cleaned KB size MB:", round(len(clean_kb.encode("utf-8")) / (1024 * 1024), 2))

    with open(CLEAN_KB_PATH, "w", encoding="utf-8") as f:
        f.write(clean_kb)

    print("Saved cleaned KB:", CLEAN_KB_PATH)

    # Free memory immediately
    del raw_kb
    del clean_kb
    gc.collect()

elapsed = time.time() - start_time
print("Cell 4 done in minutes:", round(elapsed / 60, 2))

# Quick preview without loading full file
with open(CLEAN_KB_PATH, "r", encoding="utf-8", errors="ignore") as f:
    preview = f.read(1000)

print("\nPreview:")
print(preview[:1000])

Reading raw KB: /content/Knowledge_Base.txt
Raw KB chars: 56516686
Raw KB size MB: 130.38
Cleaning KB...
Cleaned KB chars: 51994156
Cleaned KB size MB: 118.7
Saved cleaned KB: /content/drive/MyDrive/bangla_qa_070_sprint/artifacts/clean_kb.txt
Cell 4 done in minutes: 0.34

Preview:
মোঃ আব্দুল মুক্তাদির - উইকিপিডিয়া

পরিভ্রমণ
প্রধান পাতা
সম্প্রদায়ের প্রবেশদ্বার
সম্প্রদায়ের আলোচনাসভা
সাম্প্রতিক পরিবর্তন
অজানা যেকোনো পাতা
সাহায্য

অবয়ব
দান করুন

নিজস্ব সমূহ
দান করুন

পরিচ্ছেদসমূহ

সূচনা
১
জীবনের প্রথমার্ধ
২
পেশা
৩
মৃত্যু এবং উত্তরাধিকার
৪
৫
সূচিপত্র টগল করুন
মোঃ আব্দুল মুক্তাদির
১টি ভাষা
English
আন্তঃউইকি সংযোগ
নিবন্ধ
আলোচনা

কার্য

সাধারণ
সংযোগকারী পৃষ্ঠাসমূহ
সম্পর্কিত পরিবর্তন
আপলোড করুন
স্থায়ী সংযোগ
পাতার তথ্য
এই নিবন্ধটি উদ্ধৃত করুন
সংক্ষিপ্ত ইউআরএল নিন
সংক্ষিপ্ত ইউআরএল
পূর্বের পার্সারে স্যুইচ করুন
মুদ্রণ/রপ্তানি
বই তৈরি করুন
PDF ডাউনলোড

অন্যান্য প্রকল্পে
উইকিউপাত্ত আইটেম
অবয়ব

উইকিপিডিয়া, মুক্ত বিশ্বকোষ থেকে
শহীদ
ড.
মোহাম্মদ আবদুল মুক্তাদির
জন্ম
(
১৯৪০-০২-১৯
)
১৯ ফেব্রুয়ারি ১

In [ ]:
# Cell 5 — Sentence-safe KB chunking + metadata saving
# Purpose:
# - Split cleaned KB into sentence-safe chunks.
# - Use ~420 tokens with 70-token sentence-level overlap.
# - Save chunks.jsonl and chunk_metadata.parquet to Google Drive.
# RAM-safe:
# - Streams output line by line to JSONL.
# - Stores only lightweight metadata in a dataframe.
# - If chunks already exist, skips recomputation.

import re, json, gc, time
import pandas as pd
from tqdm.auto import tqdm

start_time = time.time()

CHUNK_TOKENS = CONFIG["chunk_tokens"]
CHUNK_OVERLAP = CONFIG["chunk_overlap_tokens"]
MIN_CHUNK_TOKENS = CONFIG["min_chunk_tokens"]

SENT_SPLIT_RE = re.compile(r"(?<=[।!?])\s+|(?<=\.)\s+|\n+")

def simple_tokenize_bn(text):
    """
    Token counter for chunking.
    Keeps Bangla words, English words, numbers, and punctuation tokens.
    """
    text = normalize_bn_text(text)
    return re.findall(r"[\u0980-\u09FF]+|[A-Za-z0-9]+|[^\s]", text)

def count_tokens(text):
    return len(simple_tokenize_bn(text))

def split_into_sentences(text):
    """
    Bangla-friendly sentence splitting.
    Keeps sentence boundaries as much as possible.
    """
    text = normalize_bn_text(text)
    parts = [s.strip() for s in SENT_SPLIT_RE.split(text) if s.strip()]

    sentences = []
    buffer = ""

    for s in parts:
        # Merge very short heading-like fragments with the next sentence.
        if count_tokens(s) <= 3 and not re.search(r"[।!?\.]$", s):
            buffer = (buffer + " " + s).strip()
            continue

        if buffer:
            s = (buffer + " " + s).strip()
            buffer = ""

        sentences.append(s)

    if buffer:
        sentences.append(buffer)

    return sentences

def make_sentence_chunks(text, chunk_size=420, overlap=70, min_tokens=60):
    """
    Creates sentence-safe chunks.
    Does not cut words or normal sentences in the middle.
    If a single sentence is very long, it keeps that sentence whole.
    """
    sentences = split_into_sentences(text)

    chunks = []
    cur = []
    cur_tokens = 0

    for sent in sentences:
        st = count_tokens(sent)

        # Keep overlong sentence whole instead of cutting it.
        if st > chunk_size:
            if cur:
                chunks.append(" ".join(cur).strip())
                cur = []
                cur_tokens = 0

            chunks.append(sent.strip())
            continue

        if cur_tokens + st <= chunk_size:
            cur.append(sent)
            cur_tokens += st
        else:
            if cur:
                chunks.append(" ".join(cur).strip())

            # Sentence-level overlap
            overlap_sents = []
            overlap_tokens = 0

            for prev in reversed(cur):
                pt = count_tokens(prev)
                overlap_sents.insert(0, prev)
                overlap_tokens += pt
                if overlap_tokens >= overlap:
                    break

            cur = overlap_sents + [sent]
            cur_tokens = overlap_tokens + st

    if cur:
        chunks.append(" ".join(cur).strip())

    # Remove tiny chunks
    chunks = [c for c in chunks if count_tokens(c) >= min_tokens]
    return chunks

def rough_article_split(text):
    """
    Conservative article splitting for wiki-style dumped text.
    Even if boundaries are imperfect, sentence chunking still handles it.
    """
    parts = re.split(r"\n\s*'\s*\n|\n\s*' থেকে আনীত", text)
    return [p.strip() for p in parts if len(p.strip()) > 200]

if CHUNK_PATH.exists() and CHUNK_META_PATH.exists():
    print("Chunks already exist. Skipping chunking.")
    print("CHUNK_PATH:", CHUNK_PATH)
    print("CHUNK_META_PATH:", CHUNK_META_PATH)
    chunk_meta_df = pd.read_parquet(CHUNK_META_PATH)
    print("Existing chunks:", len(chunk_meta_df))
else:
    print("Reading cleaned KB from:", CLEAN_KB_PATH)

    with open(CLEAN_KB_PATH, "r", encoding="utf-8", errors="ignore") as f:
        clean_kb = f.read()

    print("Clean KB chars:", len(clean_kb))

    articles = rough_article_split(clean_kb)
    print("Rough article/document count:", len(articles))

    meta_rows = []
    chunk_id = 0

    with open(CHUNK_PATH, "w", encoding="utf-8") as out:
        for doc_id, doc in enumerate(tqdm(articles, desc="Chunking cleaned KB")):
            doc = normalize_bn_text(doc)
            if count_tokens(doc) < MIN_CHUNK_TOKENS:
                continue

            doc_chunks = make_sentence_chunks(
                doc,
                chunk_size=CHUNK_TOKENS,
                overlap=CHUNK_OVERLAP,
                min_tokens=MIN_CHUNK_TOKENS
            )

            for local_id, chunk_text in enumerate(doc_chunks):
                tok_count = count_tokens(chunk_text)

                rec = {
                    "chunk_id": chunk_id,
                    "doc_id": doc_id,
                    "local_id": local_id,
                    "text": chunk_text,
                    "token_count": tok_count,
                    "char_count": len(chunk_text),
                }

                out.write(json.dumps(rec, ensure_ascii=False) + "\n")

                meta_rows.append({
                    "chunk_id": chunk_id,
                    "doc_id": doc_id,
                    "local_id": local_id,
                    "token_count": tok_count,
                    "char_count": len(chunk_text),
                })

                chunk_id += 1

    chunk_meta_df = pd.DataFrame(meta_rows)
    chunk_meta_df.to_parquet(CHUNK_META_PATH, index=False)

    print("Saved chunks:", CHUNK_PATH)
    print("Saved metadata:", CHUNK_META_PATH)
    print("Total chunks:", len(chunk_meta_df))

    del clean_kb
    del articles
    gc.collect()

elapsed = time.time() - start_time

print("\nChunking summary:")
print("Total chunks:", len(chunk_meta_df))
print("Avg tokens:", round(chunk_meta_df["token_count"].mean(), 2))
print("Median tokens:", round(chunk_meta_df["token_count"].median(), 2))
print("Max tokens:", int(chunk_meta_df["token_count"].max()))
print("Min tokens:", int(chunk_meta_df["token_count"].min()))
print("Elapsed minutes:", round(elapsed / 60, 2))

# Preview first chunk
with open(CHUNK_PATH, "r", encoding="utf-8") as f:
    first = json.loads(f.readline())

print("\nFirst chunk preview:")
print("chunk_id:", first["chunk_id"], "tokens:", first["token_count"])
print(first["text"][:1000])

Reading cleaned KB from: /content/drive/MyDrive/bangla_qa_070_sprint/artifacts/clean_kb.txt
Clean KB chars: 51994156
Rough article/document count: 7201


Chunking cleaned KB:   0%|          | 0/7201 [00:00<?, ?it/s]

Saved chunks: /content/drive/MyDrive/bangla_qa_070_sprint/artifacts/chunks.jsonl
Saved metadata: /content/drive/MyDrive/bangla_qa_070_sprint/artifacts/chunk_metadata.parquet
Total chunks: 32109

Chunking summary:
Total chunks: 32109
Avg tokens: 374.97
Median tokens: 411.0
Max tokens: 1234
Min tokens: 60
Elapsed minutes: 1.32

First chunk preview:
chunk_id: 0 tokens: 420
মোঃ আব্দুল মুক্তাদির - উইকিপিডিয়া পরিভ্রমণ প্রধান পাতা সম্প্রদায়ের প্রবেশদ্বার সম্প্রদায়ের আলোচনাসভা সাম্প্রতিক পরিবর্তন অজানা যেকোনো পাতা সাহায্য অবয়ব দান করুন নিজস্ব সমূহ দান করুন পরিচ্ছেদসমূহ সূচনা ১ জীবনের প্রথমার্ধ ২ পেশা ৩ মৃত্যু এবং উত্তরাধিকার ৪ ৫ সূচিপত্র টগল করুন মোঃ আব্দুল মুক্তাদির ১টি ভাষা English আন্তঃউইকি সংযোগ নিবন্ধ আলোচনা কার্য সাধারণ সংযোগকারী পৃষ্ঠাসমূহ সম্পর্কিত পরিবর্তন আপলোড করুন স্থায়ী সংযোগ পাতার তথ্য এই নিবন্ধটি উদ্ধৃত করুন সংক্ষিপ্ত ইউআরএল নিন সংক্ষিপ্ত ইউআরএল পূর্বের পার্সারে স্যুইচ করুন মুদ্রণ/রপ্তানি বই তৈরি করুন PDF ডাউনলোড অন্যান্য প্রকল্পে উইকিউপাত্ত আইটেম অবয়ব উইকিপিডিয়া, মুক্ত ব

In [ ]:
# Cell 6 — Dense embedding indexing with BGE-M3 + FAISS
# Purpose:
# - Load chunks from Drive.
# - Embed chunks using BAAI/bge-m3.
# - Build FAISS IndexFlatIP incrementally.
# - Save FAISS index + metadata to Google Drive.
# RAM-safe:
# - Does NOT store all embeddings in RAM.
# - Embeds one batch at a time.
# - Saves FAISS checkpoints every few thousand chunks.
# - If FAISS index already exists, skips recomputation.

import os, gc, json, pickle, time
import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

start_time = time.time()

DENSE_MODEL_NAME = CONFIG["dense_model"]
DENSE_BATCH_SIZE = 32   # RAM/GPU safe for BGE-M3 on T4. Increase to 48/64 only if stable.
FAISS_CHECKPOINT_EVERY = 4096

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Dense model:", DENSE_MODEL_NAME)

def iter_chunks_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

def count_jsonl_lines(path):
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for _ in f:
            n += 1
    return n

def encode_passages_bge_m3(model, texts, batch_size=32):
    # BGE-style retrieval generally works well with passage prefix.
    texts = ["passage: " + normalize_bn_text(t) for t in texts]
    embs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embs.astype("float32")

if FAISS_PATH.exists() and EMBED_META_PATH.exists():
    print("Dense FAISS index already exists. Skipping embedding.")
    print("FAISS:", FAISS_PATH)
    print("META:", EMBED_META_PATH)

    index = faiss.read_index(str(FAISS_PATH))
    with open(EMBED_META_PATH, "rb") as f:
        dense_meta = pickle.load(f)

    print("FAISS ntotal:", index.ntotal)
    print("Metadata keys:", dense_meta.keys())

else:
    total_chunks = count_jsonl_lines(CHUNK_PATH)
    print("Total chunks to embed:", total_chunks)

    model = SentenceTransformer(DENSE_MODEL_NAME, device=device)

    index = None
    chunk_ids = []
    doc_ids = []
    local_ids = []
    token_counts = []
    char_counts = []

    batch_texts = []
    batch_meta = []

    processed = 0

    pbar = tqdm(total=total_chunks, desc="Embedding chunks with BGE-M3")

    for rec in iter_chunks_jsonl(CHUNK_PATH):
        batch_texts.append(rec["text"])
        batch_meta.append(rec)

        if len(batch_texts) >= DENSE_BATCH_SIZE:
            embs = encode_passages_bge_m3(model, batch_texts, batch_size=DENSE_BATCH_SIZE)

            if index is None:
                dim = embs.shape[1]
                index = faiss.IndexFlatIP(dim)
                print("Embedding dim:", dim)

            index.add(embs)

            for m in batch_meta:
                chunk_ids.append(m["chunk_id"])
                doc_ids.append(m["doc_id"])
                local_ids.append(m["local_id"])
                token_counts.append(m["token_count"])
                char_counts.append(m["char_count"])

            processed += len(batch_texts)
            pbar.update(len(batch_texts))

            batch_texts = []
            batch_meta = []

            # Save periodic checkpoint
            if processed % FAISS_CHECKPOINT_EVERY < DENSE_BATCH_SIZE:
                faiss.write_index(index, str(FAISS_PATH))
                dense_meta = {
                    "chunk_ids": chunk_ids,
                    "doc_ids": doc_ids,
                    "local_ids": local_ids,
                    "token_counts": token_counts,
                    "char_counts": char_counts,
                    "model_name": DENSE_MODEL_NAME,
                    "processed": processed,
                    "embedding_dim": index.d,
                }
                with open(EMBED_META_PATH, "wb") as f:
                    pickle.dump(dense_meta, f)
                print("Checkpoint saved at chunks:", processed)

            del embs
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Last partial batch
    if batch_texts:
        embs = encode_passages_bge_m3(model, batch_texts, batch_size=DENSE_BATCH_SIZE)

        if index is None:
            dim = embs.shape[1]
            index = faiss.IndexFlatIP(dim)
            print("Embedding dim:", dim)

        index.add(embs)

        for m in batch_meta:
            chunk_ids.append(m["chunk_id"])
            doc_ids.append(m["doc_id"])
            local_ids.append(m["local_id"])
            token_counts.append(m["token_count"])
            char_counts.append(m["char_count"])

        processed += len(batch_texts)
        pbar.update(len(batch_texts))

        del embs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pbar.close()

    # Final save
    faiss.write_index(index, str(FAISS_PATH))

    dense_meta = {
        "chunk_ids": chunk_ids,
        "doc_ids": doc_ids,
        "local_ids": local_ids,
        "token_counts": token_counts,
        "char_counts": char_counts,
        "model_name": DENSE_MODEL_NAME,
        "processed": processed,
        "embedding_dim": index.d,
    }

    with open(EMBED_META_PATH, "wb") as f:
        pickle.dump(dense_meta, f)

    print("Final dense FAISS saved:", FAISS_PATH)
    print("Final dense metadata saved:", EMBED_META_PATH)
    print("FAISS ntotal:", index.ntotal)

    try:
        del model
    except:
        pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

elapsed = time.time() - start_time
print("Cell 6 done in minutes:", round(elapsed / 60, 2))

Device: cuda
Dense model: BAAI/bge-m3
Total chunks to embed: 32109


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding chunks with BGE-M3:   0%|          | 0/32109 [00:00<?, ?it/s]

Embedding dim: 1024
Checkpoint saved at chunks: 4096
Checkpoint saved at chunks: 8192
Checkpoint saved at chunks: 12288
Checkpoint saved at chunks: 16384
Checkpoint saved at chunks: 20480
Checkpoint saved at chunks: 24576
Checkpoint saved at chunks: 28672


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.79 GiB. GPU 0 has a total capacity of 14.56 GiB of which 5.66 GiB is free. Including non-PyTorch memory, this process has 8.90 GiB memory in use. Of the allocated memory 8.76 GiB is allocated by PyTorch, and 12.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Cell 6B — Resume BGE-M3 FAISS indexing from checkpoint after OOM
# Purpose:
# - Load existing FAISS checkpoint from Drive.
# - Load existing dense metadata checkpoint.
# - Skip already processed chunks.
# - Continue embedding remaining chunks with much smaller batch size.
# RAM-safe:
# - Uses batch size 4.
# - Truncates very long chunks for embedding only.
# - Saves checkpoint frequently.

import os, gc, json, pickle, time
import numpy as np
import torch
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

start_time = time.time()

DENSE_MODEL_NAME = CONFIG["dense_model"]
RESUME_BATCH_SIZE = 4
MAX_CHARS_FOR_EMBED = 2200
FAISS_CHECKPOINT_EVERY = 512

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Dense model:", DENSE_MODEL_NAME)

# Clean GPU memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def iter_chunks_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

def count_jsonl_lines(path):
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for _ in f:
            n += 1
    return n

def safe_truncate_for_embedding(text, max_chars=MAX_CHARS_FOR_EMBED):
    text = normalize_bn_text(text)
    if len(text) <= max_chars:
        return text
    # cut at sentence-ish boundary if possible
    cut = text[:max_chars]
    last_stop = max(cut.rfind("।"), cut.rfind("."), cut.rfind("\n"))
    if last_stop > max_chars * 0.55:
        return cut[:last_stop+1]
    return cut

def encode_passages_bge_m3(model, texts, batch_size=4):
    texts = ["passage: " + safe_truncate_for_embedding(t) for t in texts]
    embs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embs.astype("float32")

# Load checkpoint
if not FAISS_PATH.exists() or not EMBED_META_PATH.exists():
    raise FileNotFoundError("Checkpoint files not found. Need FAISS_PATH and EMBED_META_PATH to resume.")

index = faiss.read_index(str(FAISS_PATH))
with open(EMBED_META_PATH, "rb") as f:
    dense_meta = pickle.load(f)

chunk_ids = dense_meta["chunk_ids"]
doc_ids = dense_meta["doc_ids"]
local_ids = dense_meta["local_ids"]
token_counts = dense_meta["token_counts"]
char_counts = dense_meta["char_counts"]

processed = int(dense_meta.get("processed", index.ntotal))

print("Loaded checkpoint.")
print("FAISS ntotal:", index.ntotal)
print("Metadata processed:", processed)

# Safety: use the smaller of both values if mismatch
processed = min(processed, index.ntotal, len(chunk_ids))
print("Resuming after chunk count:", processed)

total_chunks = count_jsonl_lines(CHUNK_PATH)
print("Total chunks:", total_chunks)
print("Remaining:", total_chunks - processed)

model = SentenceTransformer(DENSE_MODEL_NAME, device=device)

batch_texts = []
batch_meta = []

pbar = tqdm(total=total_chunks - processed, desc="Resuming BGE-M3 embedding")

for line_idx, rec in enumerate(iter_chunks_jsonl(CHUNK_PATH)):
    if line_idx < processed:
        continue

    batch_texts.append(rec["text"])
    batch_meta.append(rec)

    if len(batch_texts) >= RESUME_BATCH_SIZE:
        try:
            embs = encode_passages_bge_m3(model, batch_texts, batch_size=RESUME_BATCH_SIZE)
        except torch.cuda.OutOfMemoryError:
            print("OOM even at batch size 4. Retrying one by one.")
            gc.collect()
            torch.cuda.empty_cache()
            embs_list = []
            for single_text in batch_texts:
                single_emb = encode_passages_bge_m3(model, [single_text], batch_size=1)
                embs_list.append(single_emb)
                gc.collect()
                torch.cuda.empty_cache()
            embs = np.vstack(embs_list).astype("float32")

        index.add(embs)

        for m in batch_meta:
            chunk_ids.append(m["chunk_id"])
            doc_ids.append(m["doc_id"])
            local_ids.append(m["local_id"])
            token_counts.append(m["token_count"])
            char_counts.append(m["char_count"])

        processed += len(batch_texts)
        pbar.update(len(batch_texts))

        batch_texts = []
        batch_meta = []

        if processed % FAISS_CHECKPOINT_EVERY < RESUME_BATCH_SIZE:
            faiss.write_index(index, str(FAISS_PATH))
            dense_meta = {
                "chunk_ids": chunk_ids,
                "doc_ids": doc_ids,
                "local_ids": local_ids,
                "token_counts": token_counts,
                "char_counts": char_counts,
                "model_name": DENSE_MODEL_NAME,
                "processed": processed,
                "embedding_dim": index.d,
            }
            with open(EMBED_META_PATH, "wb") as f:
                pickle.dump(dense_meta, f)
            print("Checkpoint saved:", processed)

        del embs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# final partial batch
if batch_texts:
    try:
        embs = encode_passages_bge_m3(model, batch_texts, batch_size=RESUME_BATCH_SIZE)
    except torch.cuda.OutOfMemoryError:
        print("OOM on final batch. Retrying one by one.")
        gc.collect()
        torch.cuda.empty_cache()
        embs_list = []
        for single_text in batch_texts:
            single_emb = encode_passages_bge_m3(model, [single_text], batch_size=1)
            embs_list.append(single_emb)
            gc.collect()
            torch.cuda.empty_cache()
        embs = np.vstack(embs_list).astype("float32")

    index.add(embs)

    for m in batch_meta:
        chunk_ids.append(m["chunk_id"])
        doc_ids.append(m["doc_id"])
        local_ids.append(m["local_id"])
        token_counts.append(m["token_count"])
        char_counts.append(m["char_count"])

    processed += len(batch_texts)
    pbar.update(len(batch_texts))

    del embs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pbar.close()

# final save
faiss.write_index(index, str(FAISS_PATH))
dense_meta = {
    "chunk_ids": chunk_ids,
    "doc_ids": doc_ids,
    "local_ids": local_ids,
    "token_counts": token_counts,
    "char_counts": char_counts,
    "model_name": DENSE_MODEL_NAME,
    "processed": processed,
    "embedding_dim": index.d,
}
with open(EMBED_META_PATH, "wb") as f:
    pickle.dump(dense_meta, f)

print("Resume complete.")
print("Final FAISS ntotal:", index.ntotal)
print("Final processed:", processed)
print("Expected total chunks:", total_chunks)
print("Done minutes:", round((time.time() - start_time) / 60, 2))

try:
    del model
except:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cuda
Dense model: BAAI/bge-m3
Loaded checkpoint.
FAISS ntotal: 28672
Metadata processed: 28672
Resuming after chunk count: 28672
Total chunks: 32109
Remaining: 3437


Resuming BGE-M3 embedding:   0%|          | 0/3437 [00:00<?, ?it/s]

Checkpoint saved: 29184
Checkpoint saved: 29696
Checkpoint saved: 30208
Checkpoint saved: 30720
Checkpoint saved: 31232
Checkpoint saved: 31744
Resume complete.
Final FAISS ntotal: 32109
Final processed: 32109
Expected total chunks: 32109
Done minutes: 16.81


In [ ]:
# Cell 7 — RAM-safe sparse/BM25 indexing with bm25s
# Purpose:
# - Build a sparse lexical retrieval index for hybrid retrieval in Sprint-2.
# - Save BM25 index + corpus metadata to Google Drive.
# RAM-safe:
# - Uses normal word tokens only, no char n-grams.
# - Saves corpus separately as JSONL.
# - Skips if index already exists.

import os, gc, json, time, pickle
from pathlib import Path
from tqdm.auto import tqdm
import bm25s

start_time = time.time()

BM25_INDEX_DIR = BM25_DIR
BM25_INDEX_DIR.mkdir(parents=True, exist_ok=True)

BM25_DONE_FLAG = BM25_INDEX_DIR / "DONE.txt"
BM25_META_PATH = BM25_INDEX_DIR / "bm25_meta.json"

def iter_chunks_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

def bm25_tokenize_bn(text):
    """
    RAM-safe lexical tokenizer.
    No char n-grams.
    """
    return bn_word_tokenize(text)

if BM25_DONE_FLAG.exists() and BM25_META_PATH.exists():
    print("BM25 index already exists. Skipping.")
    print("BM25 index dir:", BM25_INDEX_DIR)
    with open(BM25_META_PATH, "r", encoding="utf-8") as f:
        bm25_meta = json.load(f)
    print("BM25 metadata:", bm25_meta)

else:
    corpus_texts = []
    corpus_tokens = []
    corpus_ids = []

    print("Loading chunks for BM25...")

    for rec in tqdm(iter_chunks_jsonl(CHUNK_PATH), desc="Tokenizing chunks for BM25"):
        text = rec["text"]
        toks = bm25_tokenize_bn(text)

        # Skip empty lexical docs
        if not toks:
            continue

        corpus_texts.append(text)
        corpus_tokens.append(toks)
        corpus_ids.append(rec["chunk_id"])

    print("BM25 corpus size:", len(corpus_tokens))

    print("Saving BM25 corpus JSONL...")
    with open(BM25_CORPUS_PATH, "w", encoding="utf-8") as f:
        for cid, text in zip(corpus_ids, corpus_texts):
            f.write(json.dumps(
                {"chunk_id": cid, "text": text},
                ensure_ascii=False
            ) + "\n")

    print("Building bm25s index...")
    retriever = bm25s.BM25(corpus=corpus_tokens)
    retriever.index(corpus_tokens)

    print("Saving bm25s index...")
    retriever.save(str(BM25_INDEX_DIR))

    bm25_meta = {
        "corpus_size": len(corpus_tokens),
        "tokenizer": "bn_word_tokenize / normalize_for_match",
        "bm25_corpus_path": str(BM25_CORPUS_PATH),
        "bm25_index_dir": str(BM25_INDEX_DIR),
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    with open(BM25_META_PATH, "w", encoding="utf-8") as f:
        json.dump(bm25_meta, f, ensure_ascii=False, indent=2)

    with open(BM25_DONE_FLAG, "w", encoding="utf-8") as f:
        f.write("done\n")

    print("BM25 saved:", BM25_INDEX_DIR)

    del corpus_texts
    del corpus_tokens
    del corpus_ids
    del retriever
    gc.collect()

elapsed = time.time() - start_time
print("Cell 7 done in minutes:", round(elapsed / 60, 2))

Loading chunks for BM25...


Tokenizing chunks for BM25: 0it [00:00, ?it/s]

BM25 corpus size: 32109
Saving BM25 corpus JSONL...


DEBUG:bm25s:Building index from tokens


Building bm25s index...


...

BM25S Count Tokens:   0%|          | 0/32109 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/32109 [00:00<?, ?it/s]

Saving bm25s index...


Finding newlines for mmindex:   0%|          | 0.00/160M [00:00<?, ?B/s]

BM25 saved: /content/drive/MyDrive/bangla_qa_070_sprint/indexes/bm25s_index
Cell 7 done in minutes: 0.82


In [ ]:
# Cell 8 — Verify Sprint-1 saved artifacts
# Purpose:
# - Confirm that all important Sprint-1 artifacts are saved in Google Drive.
# - Check dense FAISS index, BM25 index, chunks, and metadata.
# - Save a sprint1_summary.json file for Sprint-2.
# RAM-safe:
# - Loads only metadata and FAISS header/index count.
# - Does not load all chunks into memory.

import json, pickle, time, os, gc
from pathlib import Path
import pandas as pd
import faiss

SPRINT1_SUMMARY_PATH = ARTIFACT_DIR / "sprint1_summary.json"

def file_status(path):
    path = Path(path)
    return {
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3) if path.exists() and path.is_file() else None,
    }

def dir_status(path):
    path = Path(path)
    total_size = 0
    file_count = 0
    if path.exists():
        for p in path.rglob("*"):
            if p.is_file():
                total_size += p.stat().st_size
                file_count += 1
    return {
        "path": str(path),
        "exists": path.exists(),
        "file_count": file_count,
        "size_mb": round(total_size / (1024 * 1024), 3),
    }

def count_jsonl_lines(path):
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for _ in f:
            n += 1
    return n

summary = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "project_dir": str(PROJECT_DIR),
    "artifacts": {},
    "indexes": {},
    "checks": {},
}

# Basic file checks
summary["artifacts"]["config"] = file_status(CONFIG_PATH)
summary["artifacts"]["dataset_info"] = file_status(ARTIFACT_DIR / "dataset_info.json")
summary["artifacts"]["clean_kb"] = file_status(CLEAN_KB_PATH)
summary["artifacts"]["chunks_jsonl"] = file_status(CHUNK_PATH)
summary["artifacts"]["chunk_metadata"] = file_status(CHUNK_META_PATH)

summary["indexes"]["dense_faiss"] = file_status(FAISS_PATH)
summary["indexes"]["dense_meta"] = file_status(EMBED_META_PATH)
summary["indexes"]["bm25_dir"] = dir_status(BM25_DIR)
summary["indexes"]["bm25_corpus"] = file_status(BM25_CORPUS_PATH)

# Chunk metadata check
if CHUNK_META_PATH.exists():
    chunk_meta_df = pd.read_parquet(CHUNK_META_PATH)
    summary["checks"]["chunk_meta_rows"] = int(len(chunk_meta_df))
    summary["checks"]["chunk_avg_tokens"] = float(round(chunk_meta_df["token_count"].mean(), 3))
    summary["checks"]["chunk_median_tokens"] = float(round(chunk_meta_df["token_count"].median(), 3))
    summary["checks"]["chunk_max_tokens"] = int(chunk_meta_df["token_count"].max())
else:
    summary["checks"]["chunk_meta_rows"] = None

# JSONL count check
if CHUNK_PATH.exists():
    chunk_jsonl_count = count_jsonl_lines(CHUNK_PATH)
    summary["checks"]["chunk_jsonl_rows"] = int(chunk_jsonl_count)
else:
    chunk_jsonl_count = None
    summary["checks"]["chunk_jsonl_rows"] = None

# FAISS check
if FAISS_PATH.exists():
    dense_index = faiss.read_index(str(FAISS_PATH))
    summary["checks"]["faiss_ntotal"] = int(dense_index.ntotal)
    summary["checks"]["faiss_dim"] = int(dense_index.d)
else:
    summary["checks"]["faiss_ntotal"] = None
    summary["checks"]["faiss_dim"] = None

# Dense metadata check
if EMBED_META_PATH.exists():
    with open(EMBED_META_PATH, "rb") as f:
        dense_meta = pickle.load(f)
    summary["checks"]["dense_meta_processed"] = int(dense_meta.get("processed", -1))
    summary["checks"]["dense_model"] = dense_meta.get("model_name", None)
else:
    summary["checks"]["dense_meta_processed"] = None
    summary["checks"]["dense_model"] = None

# BM25 check
bm25_done = BM25_DIR / "DONE.txt"
bm25_meta = BM25_DIR / "bm25_meta.json"
summary["checks"]["bm25_done"] = bm25_done.exists()
if bm25_meta.exists():
    with open(bm25_meta, "r", encoding="utf-8") as f:
        summary["checks"]["bm25_meta"] = json.load(f)
else:
    summary["checks"]["bm25_meta"] = None

# Consistency checks
expected_chunks = summary["checks"].get("chunk_jsonl_rows")
faiss_ntotal = summary["checks"].get("faiss_ntotal")
dense_processed = summary["checks"].get("dense_meta_processed")

summary["checks"]["dense_index_complete"] = (
    expected_chunks is not None
    and faiss_ntotal == expected_chunks
    and dense_processed == expected_chunks
)

summary["checks"]["chunk_metadata_complete"] = (
    expected_chunks is not None
    and summary["checks"].get("chunk_meta_rows") == expected_chunks
)

summary["checks"]["bm25_complete"] = bool(summary["checks"]["bm25_done"])

# Save summary
with open(SPRINT1_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Sprint-1 verification summary saved:")
print(SPRINT1_SUMMARY_PATH)

print("\nCore checks:")
print("Chunks JSONL rows:", summary["checks"]["chunk_jsonl_rows"])
print("Chunk metadata rows:", summary["checks"]["chunk_meta_rows"])
print("FAISS ntotal:", summary["checks"]["faiss_ntotal"])
print("Dense processed:", summary["checks"]["dense_meta_processed"])
print("FAISS dim:", summary["checks"]["faiss_dim"])
print("BM25 complete:", summary["checks"]["bm25_complete"])

print("\nCompletion:")
print("Dense index complete:", summary["checks"]["dense_index_complete"])
print("Chunk metadata complete:", summary["checks"]["chunk_metadata_complete"])
print("BM25 complete:", summary["checks"]["bm25_complete"])

print("\nArtifact sizes:")
for group in ["artifacts", "indexes"]:
    print("\n" + group.upper())
    for name, stat in summary[group].items():
        print(name, "=>", stat)

if (
    summary["checks"]["dense_index_complete"]
    and summary["checks"]["chunk_metadata_complete"]
    and summary["checks"]["bm25_complete"]
):
    print("\n✅ Sprint-1 complete. You can safely stop here and continue in Sprint-2.ipynb.")
else:
    print("\n⚠️ Some artifact is incomplete. Check the values above before stopping.")

gc.collect()

Sprint-1 verification summary saved:
/content/drive/MyDrive/bangla_qa_070_sprint/artifacts/sprint1_summary.json

Core checks:
Chunks JSONL rows: 32109
Chunk metadata rows: 32109
FAISS ntotal: 32109
Dense processed: 32109
FAISS dim: 1024
BM25 complete: True

Completion:
Dense index complete: True
Chunk metadata complete: True
BM25 complete: True

Artifact sizes:

ARTIFACTS
config => {'path': '/content/drive/MyDrive/bangla_qa_070_sprint/artifacts/sprint1_config.json', 'exists': True, 'size_mb': 0.0}
dataset_info => {'path': '/content/drive/MyDrive/bangla_qa_070_sprint/artifacts/dataset_info.json', 'exists': True, 'size_mb': 0.001}
clean_kb => {'path': '/content/drive/MyDrive/bangla_qa_070_sprint/artifacts/clean_kb.txt', 'exists': True, 'size_mb': 118.704}
chunks_jsonl => {'path': '/content/drive/MyDrive/bangla_qa_070_sprint/artifacts/chunks.jsonl', 'exists': True, 'size_mb': 145.498}
chunk_metadata => {'path': '/content/drive/MyDrive/bangla_qa_070_sprint/artifacts/chunk_metadata.parquet'

33